## TypedDict

In [3]:
from typing import TypedDict


class MovieDict(TypedDict):
    title: str
    year: int
    director: str
    rating: float


movie: MovieDict = {
    "title1": "盗梦空间",
    "year": 2010,
    "director": "克里斯托弗·诺兰",
    "rating": 8.8,
}

print(movie)

{'title1': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}


## Annotated

In [12]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek:deepseek-v4-pro",
    extra_body={"thinking": {"type": "disabled"}},
)

"""
使用 TypedDict 模型定义结构化输出
"""
from typing import TypedDict, Annotated


class MovieTypedDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, "电影的正式名称，例如《盗梦空间》"]
    year: Annotated[int, "电影的公映年份，使用四位数字表示"]
    director: Annotated[str, "电影导演的全名"]
    rating: Annotated[float, "电影在10分制下的评分，可包含一位小数"]


# 设置模型结构化输出
structured_llm = model.with_structured_output(MovieTypedDict)

# 调用模型并获取结构化输出
response = structured_llm.invoke("给我介绍下电影《星际穿越》")
print(type(response))
print(response)

<class 'dict'>
{'director': '克里斯托弗·诺兰', 'rating': 9.3, 'title': '星际穿越', 'year': 2014}


## 返回嵌套结构

In [26]:
from typing import TypedDict, List, Annotated


# 使用TypedDict定义嵌套结构
class Actor(TypedDict):
    """演员情况"""
    name: Annotated[str, "演员姓名"]
    role: Annotated[str, "饰演的角色"]


class Movie(TypedDict):
    """电影情况"""
    title: Annotated[str, "电影标题"]
    year: Annotated[int, "上映年份"]
    director: Annotated[str, "导演"]
    cast: Annotated[List[Actor], "演员列表"]  # 嵌套列表定义
    rating: Annotated[float, "评分"]


model = init_chat_model(model="openrouter:openai/gpt-5.6-sol")

# 设置模型结构化输出
structured_llm = model.with_structured_output(Movie)

# 调用模型并获取结构化输出
resp = structured_llm.invoke("给我介绍下电影《盗梦空间》")

# 访问嵌套数据
print(f"电影名: {resp['title']}")
print(f"上映年份: {resp['year']}")
print(f"导演: {resp['director']}")
print(f"演员列表:{resp['cast']}")
print(f"评分: {resp['rating']}")

电影名: 盗梦空间
上映年份: 2010
导演: 克里斯托弗·诺兰
演员列表:[{'name': '莱昂纳多·迪卡普里奥', 'role': '多姆·柯布'}, {'name': '约瑟夫·高登-莱维特', 'role': '亚瑟'}, {'name': '艾利奥特·佩吉', 'role': '阿里阿德涅'}, {'name': '汤姆·哈迪', 'role': '伊姆斯'}, {'name': '渡边谦', 'role': '斋藤'}, {'name': '玛丽昂·歌迪亚', 'role': '梅尔'}, {'name': '希里安·墨菲', 'role': '罗伯特·费舍'}]
评分: 8.8


## ...的使用

In [25]:
from typing import TypedDict, Annotated


class MovieDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, ..., "电影标题"]
    year: Annotated[int, ..., "电影上映年份"]
    director: Annotated[str, ..., "导演"]
    rating: Annotated[float, "电影评分，满分十分"]


model_with_structure = model.with_structured_output(MovieDict)

response = model_with_structure.invoke(
    "根据这段话抽取盗梦空间的信息，不包含的信息可以留空：盗梦空间在2010年上映，导演是克里斯托弗·诺兰。不包含的信息必须留空")

print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 0}
<class 'dict'>
